# Install Dependencies

In [ ]:
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q peft
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.6 MB/s eta 0:00:00


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(Hugging_Face_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['إيجابية', 'سلبية', 'محايدة']
    mapping = {'إيجابية': 0, 'سلبية': 1, 'محايدة':2, 'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred, digits = 4)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, model, tokenizer):
    y_pred = []
    max_length = tokenizer.model_max_length
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["Text"]
        result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
        answer = result[0]['generated_text'].split("=")[-1].lower()
        if ("سلبية" in answer
            or "سلبي" in answer
            or "سلبيّة" in answer):
            y_pred.append("سلبية")
        elif ("إيجابية" in answer
              or "إيجابي" in answer
              or "إيجابيّة" in answer):
            y_pred.append("إيجابية")
        elif ("محايدة" in answer
              or "محايد" in answer):
            y_pred.append("محايدة")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data1 = pd.read_csv('train_all.csv')
ind = pd.read_excel('Used ASAD For Training.xlsx')

data = data1.loc[ind['Unnamed: 0'].values]
data = data[["Text", "sentiment"]]

data.columns = ["Text", "sentiment"]
print(data["sentiment"].value_counts())

sentiment
neutral     16289
negative     3890
positive     3778
Name: count, dtype: int64


In [ ]:
data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو إن الأسى يأكلك بأكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا أمتلك تعريفا واضحا للإهاب بس طالما جامعة ال...,neutral


# Preprocessing

In [ ]:
data['Text'] = data['Text'].str.replace(r'[^\w\s]+', '')
data['Text'] = data['Text'].str.replace("\s+", " ", regex=True)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو إن الأسى يأكلك بأكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا أمتلك تعريفا واضحا للإهاب بس طالما جامعة ال...,neutral


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['Text'] = data['Text'].apply(normalize_arabic)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))

    return clean

In [ ]:
data['Text'] = data_cleaning(data['Text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['Text'] = data['Text'].apply(remove_ids)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
# Mapping dictionary
sentiment_map = {
    'positive': 'إيجابية',
    'negative': 'سلبية',
    'neutral': 'محايدة'
}

# Apply the mapping
data['sentiment'] = data['sentiment'].map(sentiment_map)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,إيجابية
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,إيجابية
2695,يالليل يا جامع على الود قلبين,محايدة
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,إيجابية
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,محايدة


In [ ]:
data.isnull().sum()

,0
Text,0
sentiment,0


In [ ]:
data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True)
data.shape

(23957, 2)

In [ ]:
data = data.reset_index(drop = True)

# Split Data

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate-ASAD.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    [INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر.
    قم بتصنيف مشاعر الجملة العربية التالية  بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.  [/INST]
    [{data_point["Text"]}] = [{data_point["sentiment"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر.
    قم بتصنيف مشاعر الجملة العربية التالية بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.
    [{data_point["Text"]}] =
            """.strip()

train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["Text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["Text"])

In [ ]:
y_true = test["sentiment"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["Text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

# Fine-tune Model

In [ ]:
from trl import SFTConfig

peft_config = LoraConfig(
    lora_alpha=128,
    lora_dropout=0.05,
    r=64,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1, # 4
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.01,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="Text",
    max_seq_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Adding EOS to train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/ASAD-Allam")

Epoch,Training Loss,Validation Loss
1,1.495100,1.175432


Epoch,Training Loss,Validation Loss
1,1.495100,1.175432
2,0.964100,0.742358
3,0.639300,0.420319


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/ASAD-Allam')

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=10,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

100%|██████████| 3594/3594 [39:22<00:00,  1.52it/s]

Accuracy: 0.555

f1_score:  0.6386050256370984

Classification Report:
              precision    recall  f1-score   support

           0     0.5937    0.5309    0.5605       567
           1     0.6532    0.4966    0.5642       584
           2     0.8163    0.5747    0.6745      2443
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.5551      3594
   macro avg     0.5158    0.4005    0.4498      3594
weighted avg     0.7547    0.5551    0.6386      3594


Confusion Matrix:
[[ 301   27  135  104]
 [  41  290  181   72]
 [ 165  127 1404  747]
 [   0    0    0    0]]


# Pred on Collected data

In [ ]:
collected = pd.read_excel('All Climate Change Data - All Related.xlsx')

collected.drop_duplicates(subset='text', inplace = True)
collected.dropna(inplace = True, subset='text')
collected.reset_index(drop=True, inplace = True)

In [ ]:
collected['Final Label'] = collected['Final Label'].map({
    'Negative': 'سلبية',
    'Neutral': 'محايدة',
    'Positive': 'إيجابية'
})

In [ ]:
collected['text'] = collected['text'].str.replace(r'[^\w\s]+', '')
collected['text'] = collected['text'].str.replace("\s+", " ", regex=True)

collected['text'] = collected['text'].apply(normalize_arabic)

collected['text'] = data_cleaning(collected['text'])

collected['text'] = collected['text'].apply(remove_ids)

collected.dropna(inplace = True)
collected = collected.drop_duplicates(subset = 'text')
collected = collected.rename(columns={'text': 'Text'})

collected_test = pd.DataFrame(collected.apply(generate_test_prompt, axis=1), columns=["Text"])

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(collected_test))):
    prompt = collected_test.iloc[i]["Text"]
    result = pipe(prompt, truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()

    if ("سلبية" in answer
        or "سلبي" in answer
        or "سلبيّة" in answer):
        pred.append("سلبية")
    elif ("إيجابية" in answer
          or "إيجابي" in answer
          or "إيجابيّة" in answer):
        pred.append("إيجابية")
    elif ("محايدة" in answer
          or "محايد" in answer):
        pred.append("محايدة")
    else:
        pred.append("none")

100%|██████████| 8992/8992 [1:42:44<00:00,  1.46it/s]


In [ ]:
evaluate(collected['Final Label'], pred)

Accuracy: 0.329

f1_score:  0.3303454451466727

Classification Report:
              precision    recall  f1-score   support

           0     0.6327    0.1211    0.2033      2205
           1     0.7083    0.1176    0.2018      3035
           2     0.4305    0.6226    0.5090      3752
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.3292      8992
   macro avg     0.4429    0.2153    0.2285      8992
weighted avg     0.5739    0.3292    0.3303      8992


Confusion Matrix:
[[ 267   16 1265  657]
 [  37  357 1825  816]
 [ 118  131 2336 1167]
 [   0    0    0    0]]
